In [20]:
from transformers import pipeline
from datasets import load_dataset
import evaluate
import time
import pandas as pd
import torch

In [21]:
print("Named Entity Recognition (NER)")

ner_pipe = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")

sample_text = "Roger Federer is a tennis player who was born in Basel, Switzerland. He was born on August 8, 1981"
ner_results = ner_pipe(sample_text)

print(f"Input text: {sample_text}\n")
print("Extracted Entities:")
for entity in ner_results:
    print(f" - {entity['word']} ({entity['entity_group']}): Score {entity['score']:.4f}")

Named Entity Recognition (NER)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7929.57it/s]
BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Input text: Roger Federer is a tennis player who was born in Basel, Switzerland. He was born on August 8, 1981

Extracted Entities:
 - Roger Federer (PER): Score 0.9995
 - Basel (LOC): Score 0.9953
 - Switzerland (LOC): Score 0.9997


In [22]:
print("Sentiment Analysis (Evaluation & Model Comparison)")

dataset = load_dataset("imdb", split="test[:100]")
texts = list(dataset["text"]) 
true_labels = list(dataset["label"]) # IMDB uses 0 for Negative, 1 for Positive

pipe_1 = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english", device=0 if torch.cuda.is_available() else -1)
pipe_2 = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment", device=0 if torch.cuda.is_available() else -1)

accuracy_metric = evaluate.load("accuracy")

def avg_latency(fn, inputs):
    t0 = time.time()
    outputs = list(fn(inputs))
    total_time = time.time() - t0
    return outputs, total_time / len(inputs)

# 4. DistilBERT
print("Evaluating Model 1 (DistilBERT)...")
preds_1, latency_1 = avg_latency(
    lambda x: pipe_1(x, truncation=True, max_length=512, batch_size=8), 
    texts
)
mapped_preds_1 = [1 if p['label'] == 'POSITIVE' else 0 for p in preds_1]
results_1 = accuracy_metric.compute(predictions=mapped_preds_1, references=true_labels)

# 5. Twitter RoBERTa
print("Evaluating Model 2 (Twitter RoBERTa)...")
preds_2, latency_2 = avg_latency(
    lambda x: pipe_2(x, truncation=True, max_length=512, batch_size=8), 
    texts
)
mapped_preds_2 = [1 if p['label'] == 'LABEL_2' else 0 for p in preds_2]
results_2 = accuracy_metric.compute(predictions=mapped_preds_2, references=true_labels)

comparison_data = {
    "Model ID": ["distilbert-base-uncased-finetuned-sst-2-english", "cardiffnlp/twitter-roberta-base-sentiment"],
    "Accuracy": [results_1['accuracy'], results_2['accuracy']],
    "Avg Inference Time (sec/ex)": [round(latency_1, 4), round(latency_2, 4)]
}

display(pd.DataFrame(comparison_data))

Sentiment Analysis (Evaluation & Model Comparison)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 55130.47it/s]
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluating Model 1 (DistilBERT)...
Evaluating Model 2 (Twitter RoBERTa)...


,Model ID,Accuracy,Avg Inference Time (sec/ex)
0,distilbert-base-uncased-finetuned-sst-2-english,0.89,0.0513
1,cardiffnlp/twitter-roberta-base-sentiment,0.94,0.1010


Trade Accuracy for Latency with the DistilBERT and RoBERTa models